# Prison Escape 3D - Evaluation

This notebook aims to analyze the questionaires and .jsonl logs in order to evaluate the Prison Escape 3D game.

## Load Data

In [ ]:
import plotly.express as px
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

In [ ]:
data_dir = Path("data/")
files = list(data_dir.glob("*/*.jsonl"))

dfs = []
for f in files:
    temp_df = pd.read_json(f, lines=True, encoding="utf-8-sig")
    temp_df['filename'] = f.name
    temp_df['shader'] = f.parent.name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

# expand event data
df = pd.concat([df.drop('event_data', axis=1), pd.json_normalize(df['event_data'])], axis=1)
df = pd.concat([df.drop('player_position', axis=1), pd.json_normalize(df['player_position'])], axis=1)

In [ ]:
# load and merge questionnaire data
q_shader = pd.read_csv('data/shader/questionaire_shader.csv', encoding='utf-8')
q_no_shader = pd.read_csv('data/no_shader/questionaire_no_shader.csv', encoding='utf-8')

questionnaire_df = pd.concat([q_shader, q_no_shader], ignore_index=True)

# remove .jsonl file extension
df['filename_base'] = df['filename'].str.replace('.jsonl', '', regex=False)
questionnaire_df['filename_base'] = questionnaire_df['filename_base'].str.replace('.jsonl', '', regex=False)

# ensure both columns are strings
df['filename_base'] = df['filename_base'].astype(str).str.lower()
questionnaire_df['filename_base'] = questionnaire_df['filename_base'].astype(str).str.lower()

# merge
df = df.merge(questionnaire_df, on='filename_base', how='left')

In [ ]:
df

## RQ 1: "Erleichtert die visuelle Hervorhebung von Items und Interaktionsmöglichkeiten Ausbrüche aus dem Gefängnis?"

In [ ]:
experience_col = "Wie bewertest du deine Erfahrung mit Videospielen?"

gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first()

first_gamewon_per_player.groupby(['shader', experience_col]).agg(
    num_wins=('eventType', 'size'),
).reset_index()

In [ ]:
# Group by shader AND experience
shader_experience_stats = first_gamewon_per_player.groupby(['shader', experience_col]).agg(
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

# t-Test and effect (Cohen's d) per player experience: shader vs. no_shader
ttest_rows = []
for exp_value, subset in first_gamewon_per_player.groupby(experience_col):
    shader_times = subset[subset['shader'] == 'shader']['game_time'].dropna()
    no_shader_times = subset[subset['shader'] == 'no_shader']['game_time'].dropna()

    if len(shader_times) > 0 and len(no_shader_times) > 0:
        t_stat, p_val = ttest_ind(shader_times, no_shader_times, equal_var=False, nan_policy='omit')

        n1, n2 = len(shader_times), len(no_shader_times)
        s1, s2 = shader_times.var(ddof=1), no_shader_times.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d = (shader_times.mean() - no_shader_times.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat = p_val = cohens_d = np.nan

    ttest_rows.append({
        experience_col: exp_value,
        't_statistic': t_stat,
        'p_value': p_val,
        'cohens_d': cohens_d
    })

ttest_df = pd.DataFrame(ttest_rows)
shader_experience_stats = shader_experience_stats.merge(ttest_df, on=experience_col, how='left')

print("Grouped by Shader Variant AND Player Experience:")
shader_experience_stats

In [ ]:
mental = "Geistige Anforderungen — Wie viel geistige Anstrengung war bei der Informationsaufnahme und -verarbeitung erforderlich (z.B. Denken, Entscheiden, Rechnen, Erinnern, Hinsehen, Suchen...)? War die Aufgabe leicht oder anspruchsvoll, einfach oder komplex, erforderte sie hohe Genauigkeit oder war sie fehlertolerant?"
physical = "Körperliche Anforderungen — Wie viel körperliche Aktivität war erforderlich (z.B. Ziehen, Drücken, Drehen, Steuern, Aktivieren,…)? War die Aufgabe leicht oder schwer, einfach oder anstrengend, erholsam oder mühselig?"
time = "Zeitliche Anforderungen — Wie viel Zeitdruck empfandest du hinsichtlich der Häufigkeit oder dem Takt, mit dem Aufgaben oder Aufgabenelemente auftraten? War die Abfolge langsam und geruhsam oder schnell und hektisch?"
success = "Leistung — Wie erfolgreich hast du deiner Meinung nach die vom Versuchsleiter (oder dir selbst) gesetzten Ziele erreicht? Wie zufrieden warst du mit deiner Leistung bei der Verfolgung dieser Ziele?"
effort = "Anstrengung — Wie hart musstest du arbeiten, um deinen Grad an Aufgabenerfüllung zu erreichen?"
frustration = "Frustration — Wie unsicher, entmutigt, irritiert, gestresst und\r\nverärgert (versus sicher, bestätigt, zufrieden, entspannt und zufrieden mit sich selbst) fühltest du dich während der  Aufgabe?"

nasa_tlx = df.groupby("filename")[[mental, physical, time, success, effort, frustration]].aggregate("mean")

shader_map = df[['filename', 'shader']].drop_duplicates().set_index('filename')['shader']
nasa_tlx['shader'] = nasa_tlx.index.map(shader_map)

nasa_tlx_melted = nasa_tlx.reset_index().melt(id_vars=['filename', 'shader'], var_name='Dimension', value_name='Score')
nasa_tlx_melted['Dimension'] = nasa_tlx_melted['Dimension'].str.split(' — ').str[0]
nasa_tlx_melted

In [ ]:
# Gruppiere nach Shader und Dimension, berechne Durchschnitt und Standardabweichung
nasa_stats = nasa_tlx_melted.groupby(['Dimension', 'shader'])['Score'].agg(['mean', 'std']).reset_index()

# Pivot: Dimensionen als Zeilen, Shader-Varianten als Spalten
nasa_pivot = nasa_stats.pivot(index='Dimension', columns='shader', values=['mean', 'std'])

# Flatten column names für bessere Lesbarkeit
nasa_pivot.columns = [f'{shader}_{stat}' for stat, shader in nasa_pivot.columns]

# Sortiere Spalten: erst shader, dann no_shader
col_order = ['shader_mean', 'shader_std', 'no_shader_mean', 'no_shader_std']
nasa_pivot = nasa_pivot[[col for col in col_order if col in nasa_pivot.columns]]

# Berechne t-Test, p-Wert und Cohen's d pro Dimension
test_results = []
for dimension in nasa_tlx_melted['Dimension'].unique():
    dim_data = nasa_tlx_melted[nasa_tlx_melted['Dimension'] == dimension]
    
    shader_scores = dim_data[dim_data['shader'] == 'shader']['Score'].dropna()
    no_shader_scores = dim_data[dim_data['shader'] == 'no_shader']['Score'].dropna()
    
    if len(shader_scores) > 0 and len(no_shader_scores) > 0:
        t_stat, p_val = ttest_ind(shader_scores, no_shader_scores, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(shader_scores), len(no_shader_scores)
        s1, s2 = shader_scores.var(ddof=1), no_shader_scores.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d = (shader_scores.mean() - no_shader_scores.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat = p_val = cohens_d = np.nan
    
    test_results.append({
        'Dimension': dimension,
        't_statistic': t_stat,
        'p_value': p_val,
        'cohens_d': cohens_d
    })

test_df = pd.DataFrame(test_results).set_index('Dimension')

# Füge Test-Ergebnisse zum Pivot hinzu
nasa_pivot = nasa_pivot.join(test_df)

nasa_pivot

## RQ 2: "Wie wirkt sich die visuelle Hervorhebung der Items auf das Finden der Items aus?"

In [ ]:
# Filter for ItemPickedUp events
item_pickup_df = df[df['eventType'] == 'ItemPickedUp']

# Group by shader and item_name, count occurrences
item_counts = item_pickup_df.groupby(['shader', 'item_name']).size().reset_index(name='count')

# Pivot: Items als Zeilen, Shader-Varianten als Spalten
item_counts_pivot = item_counts.pivot(index='item_name', columns='shader', values='count').fillna(0).astype(int)

# Sortiere Spalten: erst shader, dann no_shader
item_counts_pivot = item_counts_pivot[['shader', 'no_shader']]

# Benenne Spalten um
item_counts_pivot.columns = ['count_shader', 'count_no_shader']

item_counts_pivot

In [ ]:
# Durchschnittliche Zeit bis zum ersten Item-Pickup pro Item und Shader-Variante
# Filter für ItemPickedUp Events
item_pickup_data = df[df['eventType'] == 'ItemPickedUp'].copy()

# Für jeden Spieler (filename) und jedes Item: finde das erste Pickup
first_pickup_per_player_item = item_pickup_data.sort_values('game_time').groupby(['filename', 'item_name']).first().reset_index()

# Filtere Items: nur behalten, wenn mindestens 5 Spieler pro Shader-Variante das Item gefunden haben
item_player_counts = first_pickup_per_player_item.groupby(['item_name', 'shader']).size().reset_index(name='player_count')
item_min_counts = item_player_counts.groupby('item_name')['player_count'].min().reset_index()
valid_items = item_min_counts[item_min_counts['player_count'] >= 5]['item_name'].tolist()

print(f"Items mit mindestens 5 Spielern pro Shader-Variante: {len(valid_items)} von {first_pickup_per_player_item['item_name'].nunique()}")
print(f"Gefilterte Items: {valid_items}")

# Filtere Daten
first_pickup_filtered = first_pickup_per_player_item[first_pickup_per_player_item['item_name'].isin(valid_items)]

# Gruppiere nach Item und Shader, berechne Durchschnitt und Standardabweichung
item_time_stats = first_pickup_filtered.groupby(['item_name', 'shader'])['game_time'].agg(['mean', 'std', 'count']).reset_index()

# Pivot: Items als Zeilen, Shader-Varianten als Spalten
item_time_pivot = item_time_stats.pivot(index='item_name', columns='shader', values=['mean', 'std', 'count'])

# Flatten column names
item_time_pivot.columns = [f'{shader}_{stat}' for stat, shader in item_time_pivot.columns]

# Sortiere Spalten: erst shader, dann no_shader
col_order = ['shader_count', 'shader_mean', 'shader_std', 'no_shader_count', 'no_shader_mean', 'no_shader_std']
item_time_pivot = item_time_pivot[[col for col in col_order if col in item_time_pivot.columns]]

# Berechne t-Test, p-Wert und Cohen's d pro Item
test_results = []
for item_name in valid_items:
    item_data = first_pickup_filtered[first_pickup_filtered['item_name'] == item_name]
    
    shader_times = item_data[item_data['shader'] == 'shader']['game_time'].dropna()
    no_shader_times = item_data[item_data['shader'] == 'no_shader']['game_time'].dropna()
    
    if len(shader_times) >= 5 and len(no_shader_times) >= 5:
        t_stat, p_val = ttest_ind(shader_times, no_shader_times, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(shader_times), len(no_shader_times)
        s1, s2 = shader_times.var(ddof=1), no_shader_times.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d = (shader_times.mean() - no_shader_times.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat = p_val = cohens_d = np.nan
    
    test_results.append({
        'item_name': item_name,
        't_statistic': t_stat,
        'p_value': p_val,
        'cohens_d': cohens_d
    })

test_df = pd.DataFrame(test_results).set_index('item_name')

# Füge Test-Ergebnisse zum Pivot hinzu
item_time_pivot = item_time_pivot.join(test_df)

item_time_pivot

In [ ]:
# Analyse: "Die Items waren insgesamt leicht in der Spielwelt zu erkennen."
item_recognition_col = 'Die Items waren insgesamt leicht in der Spielwelt zu erkennen.'

# Pro Spieler (filename) den Durchschnittswert nehmen
item_recognition_per_player = df.groupby(['filename', 'shader'])[item_recognition_col].mean().reset_index()

# Gruppiere nach Shader-Variante
recognition_stats = item_recognition_per_player.groupby('shader')[item_recognition_col].agg(['mean', 'std', 'count']).reset_index()

# Pivot für bessere Darstellung
recognition_pivot = recognition_stats.set_index('shader').T

# Sortiere Spalten: erst shader, dann no_shader
if 'shader' in recognition_pivot.columns and 'no_shader' in recognition_pivot.columns:
    recognition_pivot = recognition_pivot[['shader', 'no_shader']]

# t-Test, p-Wert und Cohen's d
shader_scores = item_recognition_per_player[item_recognition_per_player['shader'] == 'shader'][item_recognition_col].dropna()
no_shader_scores = item_recognition_per_player[item_recognition_per_player['shader'] == 'no_shader'][item_recognition_col].dropna()

if len(shader_scores) > 0 and len(no_shader_scores) > 0:
    t_stat, p_val = ttest_ind(shader_scores, no_shader_scores, equal_var=False, nan_policy='omit')
    
    n1, n2 = len(shader_scores), len(no_shader_scores)
    s1, s2 = shader_scores.var(ddof=1), no_shader_scores.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
    cohens_d = (shader_scores.mean() - no_shader_scores.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    
    # Füge Teststatistiken hinzu
    recognition_pivot.loc['t_statistic'] = [t_stat, t_stat]
    recognition_pivot.loc['p_value'] = [p_val, p_val]
    recognition_pivot.loc['cohens_d'] = [cohens_d, cohens_d]

recognition_pivot

In [ ]:
df['Folgende Items waren einfach zu erkennen:']

In [ ]:
# Anteil einfach erkannter Items pro Spieler
items_col = 'Folgende Items waren einfach zu erkennen:'
total_items = 12

def parse_items(value: str) -> list:
    if pd.isna(value):
        return []
    items = [item.strip() for item in str(value).split(',')]
    return [item for item in items if item and item.lower() != 'keines der items']

# pro Spieler eine eindeutige Item-Liste erzeugen (falls Zeile mehrfach pro Spieler existiert)
items_per_player = (
    df.groupby(['filename', 'shader'])[items_col]
      .apply(lambda s: sorted({item for val in s.dropna() for item in parse_items(val)}))
      .reset_index(name='items_einfach')
)

items_per_player['anzahl_einfach'] = items_per_player['items_einfach'].apply(len)
items_per_player['anteil_einfach'] = items_per_player['anzahl_einfach'] / total_items

# Gruppiere nach Shader-Variante
items_stats = items_per_player.groupby('shader')['anteil_einfach'].agg(['mean', 'std', 'count']).reset_index()
items_stats.columns = ['shader', 'mean', 'std', 'count']

# t-Test, p-Wert und Cohen's d zwischen shader und no_shader
shader_scores = items_per_player[items_per_player['shader'] == 'shader']['anteil_einfach'].dropna()
no_shader_scores = items_per_player[items_per_player['shader'] == 'no_shader']['anteil_einfach'].dropna()

if len(shader_scores) > 0 and len(no_shader_scores) > 0:
    t_stat, p_val = ttest_ind(shader_scores, no_shader_scores, equal_var=False, nan_policy='omit')
    
    n1, n2 = len(shader_scores), len(no_shader_scores)
    s1, s2 = shader_scores.var(ddof=1), no_shader_scores.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
    cohens_d = (shader_scores.mean() - no_shader_scores.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
else:
    t_stat = p_val = cohens_d = np.nan

# Füge Test-Ergebnisse hinzu
items_stats['t_statistic'] = t_stat
items_stats['p_value'] = p_val
items_stats['cohens_d'] = cohens_d

print("Anteil einfach erkannter Items pro Shader-Variante:")
items_stats

In [ ]:
# Anteil mühsam erkannter Items pro Spieler
items_difficult_col = 'Folgende Items waren mühsam zu erkennen:'

# pro Spieler eine eindeutige Item-Liste erzeugen
difficult_items_per_player = (
    df.groupby(['filename', 'shader'])[items_difficult_col]
      .apply(lambda s: sorted({item for val in s.dropna() for item in parse_items(val)}))
      .reset_index(name='items_mühsam')
)

difficult_items_per_player['anzahl_mühsam'] = difficult_items_per_player['items_mühsam'].apply(len)
difficult_items_per_player['anteil_mühsam'] = difficult_items_per_player['anzahl_mühsam'] / total_items

# Gruppiere nach Shader-Variante und berechne statistische Tests
# für den Anteil mühsam erkannter Items

# Gruppiere nach Shader-Variante
difficult_items_stats = difficult_items_per_player.groupby('shader')['anteil_mühsam'].agg(['mean', 'std', 'count']).reset_index()
difficult_items_stats.columns = ['shader', 'mean', 'std', 'count']

# t-Test, p-Wert und Cohen's d zwischen shader und no_shader
shader_difficult_scores = difficult_items_per_player[difficult_items_per_player['shader'] == 'shader']['anteil_mühsam'].dropna()
no_shader_difficult_scores = difficult_items_per_player[difficult_items_per_player['shader'] == 'no_shader']['anteil_mühsam'].dropna()

if len(shader_difficult_scores) > 0 and len(no_shader_difficult_scores) > 0:
    t_stat_diff, p_val_diff = ttest_ind(shader_difficult_scores, no_shader_difficult_scores, equal_var=False, nan_policy='omit')
    
    n1, n2 = len(shader_difficult_scores), len(no_shader_difficult_scores)
    s1, s2 = shader_difficult_scores.var(ddof=1), no_shader_difficult_scores.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
    cohens_d_diff = (shader_difficult_scores.mean() - no_shader_difficult_scores.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
else:
    t_stat_diff = p_val_diff = cohens_d_diff = np.nan

# Füge Test-Ergebnisse hinzu
difficult_items_stats['t_statistic'] = t_stat_diff
difficult_items_stats['p_value'] = p_val_diff
difficult_items_stats['cohens_d'] = cohens_d_diff

print("Anteil mühsam erkannter Items pro Shader-Variante:")
difficult_items_stats

## RQ 3: "Spiegelt sich die eigene Einschätzung der Risikoneigung von Spielern in ihrem Spielverhalten wider?"

In [ ]:
df.columns

In [ ]:
risk_tendency_col = 'Wie schätzt du deine Risikoneigung ein?'
risky_actions_col = 'Ich habe bewusst riskante Aktionen in Kauf genommen (z. B. Items in der Nähe von Wärtern benutzt).'

# Pro Spieler den Durchschnittswert für riskante Aktionen nehmen
risk_analysis = df.groupby(['filename', risk_tendency_col])[[risky_actions_col]].mean().reset_index()
risk_analysis = risk_analysis.dropna(subset=[risky_actions_col])

# Gruppiere nach Risikoneigung
risk_stats = risk_analysis.groupby(risk_tendency_col)[risky_actions_col].agg(['mean', 'std', 'count']).reset_index()

print("Riskante Aktionen gruppiert nach Risikoneigung:")

# Statistischer Test zwischen risikoneigungsgruppen
# Extrahiere die Risikoneigungswerte (sortiert)
risk_levels = sorted(risk_analysis[risk_tendency_col].unique())

# t-Test zwischen extremen Gruppen (niedrigste vs. höchste Risikoneigung)
if len(risk_levels) >= 2:
    low_risk = risk_analysis[risk_analysis[risk_tendency_col] == risk_levels[0]][risky_actions_col].dropna()
    high_risk = risk_analysis[risk_analysis[risk_tendency_col] == risk_levels[-1]][risky_actions_col].dropna()
    
    if len(low_risk) > 0 and len(high_risk) > 0:
        t_stat_risk, p_val_risk = ttest_ind(low_risk, high_risk, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(low_risk), len(high_risk)
        s1, s2 = low_risk.var(ddof=1), high_risk.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d_risk = (low_risk.mean() - high_risk.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat_risk = p_val_risk = cohens_d_risk = np.nan
else:
    t_stat_risk = p_val_risk = cohens_d_risk = np.nan

# Füge Test-Ergebnisse zum DataFrame hinzu
risk_stats['t_statistic'] = t_stat_risk
risk_stats['p_value'] = p_val_risk
risk_stats['cohens_d'] = cohens_d_risk

print("Vergleich: Niedrigste vs. Höchste Risikoneigung")
print(f"t-Statistic: {t_stat_risk}")
print(f"p-Value: {p_val_risk}")
print(f"Cohen's d: {cohens_d_risk}")
print("\n")
risk_stats

In [ ]:
faster_riskier_col = 'Ich habe eher den schnelleren, riskanteren Ansatz gewählt als den langsameren, sicheren.'

# Pro Spieler den Durchschnittswert nehmen
risk_approach_analysis = df.groupby(['filename', risk_tendency_col])[[faster_riskier_col]].mean().reset_index()
risk_approach_analysis = risk_approach_analysis.dropna(subset=[faster_riskier_col])

# Gruppiere nach Risikoneigung
risk_approach_stats = risk_approach_analysis.groupby(risk_tendency_col)[faster_riskier_col].agg(['mean', 'std', 'count']).reset_index()

# t-Test zwischen risikoneigungsgruppen
if len(risk_levels) >= 2:
    low_risk_approach = risk_approach_analysis[risk_approach_analysis[risk_tendency_col] == risk_levels[0]][faster_riskier_col].dropna()
    high_risk_approach = risk_approach_analysis[risk_approach_analysis[risk_tendency_col] == risk_levels[-1]][faster_riskier_col].dropna()
    
    if len(low_risk_approach) > 0 and len(high_risk_approach) > 0:
        t_stat_approach, p_val_approach = ttest_ind(low_risk_approach, high_risk_approach, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(low_risk_approach), len(high_risk_approach)
        s1, s2 = low_risk_approach.var(ddof=1), high_risk_approach.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d_approach = (low_risk_approach.mean() - high_risk_approach.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat_approach = p_val_approach = cohens_d_approach = np.nan
else:
    t_stat_approach = p_val_approach = cohens_d_approach = np.nan

# Füge Test-Ergebnisse hinzu
risk_approach_stats['t_statistic'] = t_stat_approach
risk_approach_stats['p_value'] = p_val_approach
risk_approach_stats['cohens_d'] = cohens_d_approach

print("Schnellerer, riskanterer Ansatz gruppiert nach Risikoneigung:")
print("Vergleich: Niedrigste vs. Höchste Risikoneigung")
print(f"t-Statistic: {t_stat_approach}")
print(f"p-Value: {p_val_approach}")
print(f"Cohen's d: {cohens_d_approach}")
print("\n")
risk_approach_stats

In [ ]:
scouting_col = 'Ich habe häufig abgewartet/gescoutet, um Risiken zu minimieren (z. B. Wärter beobachten).'

# Pro Spieler den Durchschnittswert nehmen
scouting_analysis = df.groupby(['filename', risk_tendency_col])[[scouting_col]].mean().reset_index()
scouting_analysis = scouting_analysis.dropna(subset=[scouting_col])

# Gruppiere nach Risikoneigung
scouting_stats = scouting_analysis.groupby(risk_tendency_col)[scouting_col].agg(['mean', 'std', 'count']).reset_index()

# t-Test zwischen risikoneigungsgruppen
if len(risk_levels) >= 2:
    low_risk_scouting = scouting_analysis[scouting_analysis[risk_tendency_col] == risk_levels[0]][scouting_col].dropna()
    high_risk_scouting = scouting_analysis[scouting_analysis[risk_tendency_col] == risk_levels[-1]][scouting_col].dropna()
    
    if len(low_risk_scouting) > 0 and len(high_risk_scouting) > 0:
        t_stat_scouting, p_val_scouting = ttest_ind(low_risk_scouting, high_risk_scouting, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(low_risk_scouting), len(high_risk_scouting)
        s1, s2 = low_risk_scouting.var(ddof=1), high_risk_scouting.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d_scouting = (low_risk_scouting.mean() - high_risk_scouting.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat_scouting = p_val_scouting = cohens_d_scouting = np.nan
else:
    t_stat_scouting = p_val_scouting = cohens_d_scouting = np.nan

# Füge Test-Ergebnisse hinzu
scouting_stats['t_statistic'] = t_stat_scouting
scouting_stats['p_value'] = p_val_scouting
scouting_stats['cohens_d'] = cohens_d_scouting

print("Abwarten/Scouting zum Risikominimierung gruppiert nach Risikoneigung:")
print("Vergleich: Niedrigste vs. Höchste Risikoneigung")
print(f"t-Statistic: {t_stat_scouting}")
print(f"p-Value: {p_val_scouting}")
print(f"Cohen's d: {cohens_d_scouting}")
print("\n")
scouting_stats

In [ ]:
# PlayerCaught Events nach Risikoneigung gruppieren
risk_tendency_col = 'Wie schätzt du deine Risikoneigung ein?'

# Zähle PlayerCaught Events pro Spieler (direkt aus dem df, das bereits die Risikoneigung enthält)
caught_per_player = (
    df.groupby(['filename', risk_tendency_col])['eventType']
    .apply(lambda x: (x == 'PlayerCaught').sum())
    .reset_index(name='num_caught')
)

# Gruppiere nach Risikoneigung für Statistik
caught_stats = caught_per_player.groupby(risk_tendency_col)['num_caught'].agg(['mean', 'std', 'count']).reset_index()

# t-Test zwischen niedrigster und höchster Risikoneigung
risk_levels = sorted(caught_per_player[risk_tendency_col].dropna().unique())
if len(risk_levels) >= 2:
    low_risk = caught_per_player[caught_per_player[risk_tendency_col] == risk_levels[0]]['num_caught']
    high_risk = caught_per_player[caught_per_player[risk_tendency_col] == risk_levels[-1]]['num_caught']
    
    if len(low_risk) > 0 and len(high_risk) > 0:
        t_stat_caught, p_val_caught = ttest_ind(low_risk, high_risk, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(low_risk), len(high_risk)
        s1, s2 = low_risk.var(ddof=1), high_risk.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d_caught = (low_risk.mean() - high_risk.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat_caught = p_val_caught = cohens_d_caught = np.nan
else:
    t_stat_caught = p_val_caught = cohens_d_caught = np.nan

# Füge Test-Ergebnisse hinzu
caught_stats['t_statistic'] = t_stat_caught
caught_stats['p_value'] = p_val_caught
caught_stats['cohens_d'] = cohens_d_caught

print("PlayerCaught pro Spieler gruppiert nach Risikoneigung:")
print("Vergleich: Niedrigste vs. Höchste Risikoneigung")
print(f"t-Statistic: {t_stat_caught}")
print(f"p-Value: {p_val_caught}")
print(f"Cohen's d: {cohens_d_caught}")
print("\n")
caught_stats

In [ ]:
# SuspiciousEventTriggered Events nach Risikoneigung gruppieren
risk_tendency_col = 'Wie schätzt du deine Risikoneigung ein?'

# Zähle SuspiciousEventTriggered Events pro Spieler
suspicious_per_player = (
    df.groupby(['filename', risk_tendency_col])['eventType']
    .apply(lambda x: (x == 'SuspiciousEventTriggered').sum())
    .reset_index(name='num_suspicious')
)

# Gruppiere nach Risikoneigung für Statistik
suspicious_stats = suspicious_per_player.groupby(risk_tendency_col)['num_suspicious'].agg(['mean', 'std', 'count']).reset_index()

# t-Test zwischen niedrigster und höchster Risikoneigung
risk_levels = sorted(suspicious_per_player[risk_tendency_col].dropna().unique())
if len(risk_levels) >= 2:
    low_risk = suspicious_per_player[suspicious_per_player[risk_tendency_col] == risk_levels[0]]['num_suspicious']
    high_risk = suspicious_per_player[suspicious_per_player[risk_tendency_col] == risk_levels[-1]]['num_suspicious']
    
    if len(low_risk) > 0 and len(high_risk) > 0:
        t_stat_suspicious, p_val_suspicious = ttest_ind(low_risk, high_risk, equal_var=False, nan_policy='omit')
        
        n1, n2 = len(low_risk), len(high_risk)
        s1, s2 = low_risk.var(ddof=1), high_risk.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d_suspicious = (low_risk.mean() - high_risk.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat_suspicious = p_val_suspicious = cohens_d_suspicious = np.nan
else:
    t_stat_suspicious = p_val_suspicious = cohens_d_suspicious = np.nan

# Füge Test-Ergebnisse hinzu
suspicious_stats['t_statistic'] = t_stat_suspicious
suspicious_stats['p_value'] = p_val_suspicious
suspicious_stats['cohens_d'] = cohens_d_suspicious

print("SuspiciousEventTriggered pro Spieler gruppiert nach Risikoneigung:")
print("Vergleich: Niedrigste vs. Höchste Risikoneigung")
print(f"t-Statistic: {t_stat_suspicious}")
print(f"p-Value: {p_val_suspicious}")
print(f"Cohen's d: {cohens_d_suspicious}")
print("\n")
suspicious_stats

In [ ]:
# Erste Player Wins nach Risikoneigung gruppieren
risk_tendency_col = 'Wie schätzt du deine Risikoneigung ein?'

# erstes GameWon pro Spieler
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first().reset_index()

# Gruppiere nach Risikoneigung
wins_by_risk = first_gamewon_per_player.groupby(risk_tendency_col).agg(
    num_wins=('eventType', 'size'),
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

wins_by_risk

## Misc

### Game Events

In [ ]:
# avg game_time for first GameWon Event per player (filename)
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first()
avg_game_time_first_gamewon = first_gamewon_per_player['game_time'].mean()

# group by shader variant
shader_stats = first_gamewon_per_player.groupby('shader').agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean')
)

print("Grouped by Shader Variant:")
shader_stats

In [ ]:
# Group by shader variant AND player experience
experience_col = "Wie bewertest du deine Erfahrung mit Videospielen?"

# Get first game won per player with shader and experience info
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first().reset_index()

# Group by shader AND experience
shader_experience_stats = first_gamewon_per_player.groupby(['shader', experience_col]).agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

# t-Test and effect (Cohen's d) per player experience: shader vs. no_shader
ttest_rows = []
for exp_value, subset in first_gamewon_per_player.groupby(experience_col):
    shader_times = subset[subset['shader'] == 'shader']['game_time'].dropna()
    no_shader_times = subset[subset['shader'] == 'no_shader']['game_time'].dropna()

    if len(shader_times) > 0 and len(no_shader_times) > 0:
        t_stat, p_val = ttest_ind(shader_times, no_shader_times, equal_var=False, nan_policy='omit')

        n1, n2 = len(shader_times), len(no_shader_times)
        s1, s2 = shader_times.var(ddof=1), no_shader_times.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d = (shader_times.mean() - no_shader_times.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat = p_val = cohens_d = np.nan

    ttest_rows.append({
        experience_col: exp_value,
        't_statistic': t_stat,
        'p_value': p_val,
        'cohens_d': cohens_d
    })

ttest_df = pd.DataFrame(ttest_rows)
shader_experience_stats = shader_experience_stats.merge(ttest_df, on=experience_col, how='left')

print("Grouped by Shader Variant AND Player Experience:")
shader_experience_stats

In [ ]:
# Overall comparison Shader vs. No-Shader (without player experience grouping)
# Uses the same first game win data but without splitting by experience
overall_shader_stats = first_gamewon_per_player.groupby('shader').agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

shader_times_all = first_gamewon_per_player[first_gamewon_per_player['shader'] == 'shader']['game_time'].dropna()
no_shader_times_all = first_gamewon_per_player[first_gamewon_per_player['shader'] == 'no_shader']['game_time'].dropna()

if len(shader_times_all) > 0 and len(no_shader_times_all) > 0:
    t_stat_overall, p_val_overall = ttest_ind(shader_times_all, no_shader_times_all, equal_var=False, nan_policy='omit')
    n1, n2 = len(shader_times_all), len(no_shader_times_all)
    s1, s2 = shader_times_all.var(ddof=1), no_shader_times_all.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
    cohens_d_overall = (shader_times_all.mean() - no_shader_times_all.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
else:
    t_stat_overall = p_val_overall = cohens_d_overall = np.nan

overall_shader_stats['t_statistic'] = t_stat_overall
overall_shader_stats['p_value'] = p_val_overall
overall_shader_stats['cohens_d'] = cohens_d_overall

print("Comparison Shader vs. No-Shader (without player experience):")
overall_shader_stats

In [ ]:
# count PlayerCaught events
player_caught_df = df[df['eventType'] == 'PlayerCaught']
player_caught_stats = player_caught_df.groupby('shader').agg(
    num_caught=('eventType', 'size')
)

print("PlayerCaught events per shader variant:")
player_caught_stats

In [ ]:
experience_col = "Wie bewertest du deine Erfahrung mit Videospielen?"

# filter for first game won events
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_wins = gamewon_df.groupby('filename').first().reset_index()

# Scatter Plot
fig = px.scatter(first_wins, x=experience_col, y='game_time',
                 title='Spielzeit bis zum ersten Win vs. Spielererfahrung',
                 labels={experience_col: 'Spielerfahrung (1-6)', 'game_time': 'Spielzeit bis Win (Sekunden)'})
fig.show()

### Items

In [ ]:
# Filter for ItemPickedUp events
item_pickup_df = df[df['eventType'] == 'ItemPickedUp']

# Group by shader and item_name, count occurrences
item_counts = item_pickup_df.groupby(['shader', 'item_name']).size().reset_index(name='count')

# Create bar chart
fig = px.bar(item_counts, x='item_name', y='count', color='shader', barmode='group', 
             title='Häufigkeit der Item-Aufnahmen pro Shader-Variante',
             labels={'item_name': 'Item Name', 'count': 'Anzahl Aufnahmen', 'shader': 'Shader Variante'})

fig.show()

### Spreadsheet

In [ ]:
# SpreadsheetRouteClicked Events
route_clicked_df = df[df['eventType'] == 'SpreadsheetRouteClicked']
route_counts = route_clicked_df['route_name'].value_counts()

fig = px.bar(route_counts, x=route_counts.index, y=route_counts.values, 
             title='Häufigkeit der Route Names für SpreadsheetRouteClicked',
             labels={'x': 'Route Name', 'y': 'Häufigkeit'})
fig.show()

### Suspicious

In [ ]:
# SuspiciousEventTriggered Events
suspicious_df = df[df['eventType'] == 'SuspiciousEventTriggered']
reason_counts = suspicious_df['reason'].value_counts()

fig = px.bar(reason_counts, x=reason_counts.index, y=reason_counts.values, 
             title='Häufigkeit der Reasons für SuspiciousEventTriggered',
             labels={'x': 'Reason', 'y': 'Häufigkeit'})
fig.show()

### NASA TLX

In [ ]:
mental = "Geistige Anforderungen — Wie viel geistige Anstrengung war bei der Informationsaufnahme und -verarbeitung erforderlich (z.B. Denken, Entscheiden, Rechnen, Erinnern, Hinsehen, Suchen...)? War die Aufgabe leicht oder anspruchsvoll, einfach oder komplex, erforderte sie hohe Genauigkeit oder war sie fehlertolerant?"
physical = "Körperliche Anforderungen — Wie viel körperliche Aktivität war erforderlich (z.B. Ziehen, Drücken, Drehen, Steuern, Aktivieren,…)? War die Aufgabe leicht oder schwer, einfach oder anstrengend, erholsam oder mühselig?"
time = "Zeitliche Anforderungen — Wie viel Zeitdruck empfandest du hinsichtlich der Häufigkeit oder dem Takt, mit dem Aufgaben oder Aufgabenelemente auftraten? War die Abfolge langsam und geruhsam oder schnell und hektisch?"
success = "Leistung — Wie erfolgreich hast du deiner Meinung nach die vom Versuchsleiter (oder dir selbst) gesetzten Ziele erreicht? Wie zufrieden warst du mit deiner Leistung bei der Verfolgung dieser Ziele?"
effort = "Anstrengung — Wie hart musstest du arbeiten, um deinen Grad an Aufgabenerfüllung zu erreichen?"
frustration = "Frustration — Wie unsicher, entmutigt, irritiert, gestresst und\r\nverärgert (versus sicher, bestätigt, zufrieden, entspannt und zufrieden mit sich selbst) fühltest du dich während der  Aufgabe?"

nasa_tlx = df.groupby("filename")[[mental, physical, time, success, effort, frustration]].aggregate("mean")

shader_map = df[['filename', 'shader']].drop_duplicates().set_index('filename')['shader']
nasa_tlx['shader'] = nasa_tlx.index.map(shader_map)

nasa_tlx_melted = nasa_tlx.reset_index().melt(id_vars=['filename', 'shader'], var_name='Dimension', value_name='Score')
nasa_tlx_melted['Dimension'] = nasa_tlx_melted['Dimension'].str.split(' — ').str[0]

fig = px.box(nasa_tlx_melted, x='Dimension', y='Score', color='shader',
             title='NASA-TLX Scores pro Dimension und Shader-Variante',
             labels={'Dimension': 'NASA-TLX Dimension', 'Score': 'Score', 'shader': 'Shader Variante'})
fig.show()

In [ ]:
# Durchschnittliche Zeit bis Item-Aufnahme pro Item und Shader-Variante
# Filtere nur ItemPickedUp Events
item_pickup_data = df[df['eventType'] == 'ItemPickedUp'].copy()

# Für jedes Spiel (session_id) und Item: finde das erste Pickup
first_pickup_per_item = item_pickup_data.sort_values('game_time').groupby(['filename', 'item_name']).first().reset_index()

# Gruppiere nach Item und Shader-Variante, berechne Durchschnitte für Zeit und Position
avg_stats = first_pickup_per_item.groupby(['shader', 'item_name']).agg({
    'game_time': ['mean', 'count'],
    'x': 'mean',
    'y': 'mean',
    'z': 'mean'
}).reset_index()

# Flatten column names
avg_stats.columns = ['Shader', 'Item', 'Durchschn. Zeit (s)', 'Anzahl Items', 'Durchschn. X', 'Durchschn. Y', 'Durchschn. Z']

# Runde Positionen auf 2 Dezimalstellen
avg_stats['Durchschn. X'] = avg_stats['Durchschn. X'].round(2)
avg_stats['Durchschn. Y'] = avg_stats['Durchschn. Y'].round(2)
avg_stats['Durchschn. Z'] = avg_stats['Durchschn. Z'].round(2)
avg_stats['Durchschn. Zeit (s)'] = avg_stats['Durchschn. Zeit (s)'].round(2)

# Sortiere nach Shader und durchschnittlicher Zeit
avg_stats = avg_stats.sort_values(['Shader', 'Anzahl Items'], ascending=[True, False])

print("Durchschnittliche Zeit und Position bis zur Aufnahme pro Item und Shader-Variante:")
avg_stats

In [ ]:
# Boxplot: Item Pickup Zeiten pro Item und Shader-Variante
fig = px.box(first_pickup_per_item, 
             x='item_name', 
             y='game_time', 
             color='shader',
             title='Verteilung der Item Pickup Zeiten pro Item und Shader-Variante',
             labels={'item_name': 'Item Name', 'game_time': 'Zeit bis Pickup (Sekunden)', 'shader': 'Shader Variante'})

fig.update_traces(boxmean=True)
fig.show()